In [ ]:
!pip install -q ragas langchain langchain-openai langchain-community langchain-mistralai sentence-transformers psycopg2-binary pgvector langchain_huggingface langchain-postgres

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 138.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 140.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB

In [ ]:
import sys
import types

class DummyVertexAI: pass
class DummyChatVertexAI: pass

dummy_llms = types.ModuleType("langchain_community.llms")
dummy_llms.VertexAI = DummyVertexAI
sys.modules["langchain_community.llms"] = dummy_llms

dummy_chat_models = types.ModuleType("langchain_community.chat_models")
dummy_chat_models.ChatVertexAI = DummyChatVertexAI
sys.modules["langchain_community.chat_models"] = dummy_chat_models

dummy_chat_vertexai = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertexai.ChatVertexAI = DummyChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertexai

dummy_llms_vertexai = types.ModuleType("langchain_community.llms.vertexai")
dummy_llms_vertexai.VertexAI = DummyVertexAI
sys.modules["langchain_community.llms.vertexai"] = dummy_llms_vertexai

In [ ]:
import os
import pandas as pd
import nest_asyncio
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.pgvector import PGVector
from ragas import EvaluationDataset
from ragas import evaluate
from google.colab import userdata, drive

# Apply nest_asyncio to support asynchronous evaluation processing
nest_asyncio.apply()

# Configure environment keys and database connection strings
NEON_CONNECTION_STRING = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# Define drive storage paths
DRIVE_PATH = '/content/drive/MyDrive/'
DATASET_PATH = DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv'

# Mount Google Drive for dataset access
drive.mount('/content/drive')

/tmp/ipykernel_5676/1482515593.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Mounted at /content/drive


## Configuration and setup

In [ ]:
# ==========================================
# CELL 1: CONFIGURATION & SETUP
# ==========================================
import os
import json
import pandas as pd
import time
import nest_asyncio
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from ragas.run_config import RunConfig
from google.colab import userdata, drive

# Enable async processing for Ragas
nest_asyncio.apply()


# Load the 200 Question Dataset
df = pd.read_csv(DATASET_PATH)

# 2. INITIALIZE GENERATOR & RAGAS JUDGE
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)
judge_llm = LangchainLLMWrapper(llm)

# Ragas Math Evaluator (Locked to BGE-M3 for consistency)
bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device': 'cpu'})
ragas_embeddings = LangchainEmbeddingsWrapper(bge_embeddings)

# 3. PROMPT TEMPLATES (Translate-then-Retrieve)
translation_prompt = PromptTemplate(
    template="You are an expert medical translator. Translate the following English medical question into precise German clinical terminology for searching guidelines. Output ONLY the translation and nothing else.\n\nQuestion:\n{question}",
    input_variables=["question"]
)

qa_prompt = PromptTemplate(
    template="You are an expert medical AI. Read the German clinical guidelines and answer the medical question in ENGLISH.\nUse ONLY the provided German context to formulate your answer.\n\nContext (German):\n{context}\n\nQuestion (English):\n{question}\n\nAnswer (English):",
    input_variables=["context", "question"]
)

# 4. DEFINE THE MODELS TO TEST
medical_configs = [
    {
        "name": "awmf_crosslingual_biobert_k10",
        "collection": "awmf_baseline_biobert",
        "model_repo": "dmis-lab/biobert-v1.1"
    },
    {
        "name": "awmf_crosslingual_pubmedbert_k10",
        "collection": "awmf_baseline_pubmedbert",
        "model_repo": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
    }
]

print("Setup complete! Models and prompts are configured.")

/tmp/ipykernel_5676/2061263543.py:11: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_5676/2061263543.py:11: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_5676/2061263543.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_pre

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Setup complete! Models and prompts are configured.


/tmp/ipykernel_5676/2061263543.py:39: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(bge_embeddings)


## Generation Loop

In [ ]:
# ==========================================
# CELL 2: GENERATION (Retrieval & QA)
# ==========================================

for config in medical_configs:
    model_name = config["name"]
    collection_name = config["collection"]
    output_file = f"{DRIVE_PATH}backup_{model_name}_gpt4_results.json"

    print(f"\n" + "="*60)
    print(f"Loading Medical Embedder: {config['model_repo']}")
    print("="*60)

    # Load specific medical embedder
    current_embeddings = HuggingFaceEmbeddings(
        model_name=config["model_repo"],
        model_kwargs={'device': 'cpu'}
    )

    # Connect to the specific Postgres collection
    vector_store = PGVector(
        embeddings=current_embeddings,
        collection_name=collection_name,
        connection=NEON_CONNECTION_STRING,
        use_jsonb=True
    )
    retriever = vector_store.as_retriever(search_kwargs={"k": 10}) # Locked at k=10

    results = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

    # Run the 200 Questions
    for index, row in df.iterrows():
        english_question = row['English_Open_Question']
        english_ground_truth = row['English_Correct_Text']

        try:
            # A: Translate query to German
            formatted_trans_prompt = translation_prompt.format(question=english_question)
            german_search_query = llm.invoke(formatted_trans_prompt).content.strip()

            # B: Retrieve domain-specific German chunks
            retrieved_docs = retriever.invoke(german_search_query)
            contexts = [doc.page_content for doc in retrieved_docs]
            context_string = "\n\n".join(contexts)

            # C: Generate English Clinical Answer
            formatted_qa_prompt = qa_prompt.format(context=context_string, question=english_question)
            response_msg = llm.invoke(formatted_qa_prompt)

            # Append and Save
            results["question"].append(english_question)
            results["answer"].append(response_msg.content)
            results["contexts"].append(contexts)
            results["ground_truth"].append(english_ground_truth)

            with open(output_file, 'w') as f:
                json.dump(results, f)

            if (index + 1) % 20 == 0:
                print(f"[{model_name}] Progress: {index + 1}/{len(df)} generated.")
            time.sleep(0.4) # Avoid rate limits

        except Exception as e:
            print(f"Error at index {index} for {model_name}: {e}")
            continue

    print(f"Generation fully completed for {model_name}!")


Loading Medical Embedder: dmis-lab/biobert-v1.1


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

[awmf_crosslingual_pubmedbert_k10] Progress: 20/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 40/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 60/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 80/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 100/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 120/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 140/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 160/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 180/200 generated.
[awmf_crosslingual_pubmedbert_k10] Progress: 200/200 generated.
Generation fully completed for awmf_crosslingual_pubmedbert_k10!


## Evaluation

In [ ]:
# ==========================================
# CELL 3: EVALUATION (Grading with Ragas)
# ==========================================

evaluation_features = Features({
    "question": Value("string"),
    "answer": Value("string"),
    "contexts": Sequence(Value("string")),
    "ground_truth": Value("string"),
})

for config in medical_configs:
    model_name = config["name"]
    input_file = f"{DRIVE_PATH}backup_{model_name}_gpt4_results.json"

    print(f"\n" + "="*60)
    print(f"Starting Evaluation for: {model_name}")
    print("="*60)

    if not os.path.exists(input_file):
        print(f"Could not find {input_file}. Did the generation cell finish?")
        continue

    # Load generation data
    with open(input_file, 'r') as f:
        data = json.load(f)

    eval_dataset = Dataset.from_dict(data, features=evaluation_features)

    try:
        # Run Ragas grading
        eval_results = evaluate(
            dataset=eval_dataset,
            metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
            llm=judge_llm,
            embeddings=ragas_embeddings,
            run_config=RunConfig(timeout=300, max_workers=2, max_retries=5)
        )

        # Convert to Pandas and save
        res_df = eval_results.to_pandas()
        res_df['model'] = model_name

        output_path = f"{DRIVE_PATH}FINAL_EVALUATION_{model_name}_gpt4.csv"
        res_df.to_csv(output_path, index=False)

        print(f"EVALUATION COMPLETE FOR {model_name} -> Saved to Drive!")
        print(res_df[['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']].mean().round(3))

    except Exception as e:
        print(f"Error during evaluation grading for {model_name}: {e}")


Starting Evaluation for: awmf_crosslingual_biobert_k10


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

EVALUATION COMPLETE FOR awmf_crosslingual_biobert_k10 -> Saved to Drive!
context_precision    0.000
context_recall       0.015
faithfulness         0.084
answer_relevancy     0.516
dtype: float64

Starting Evaluation for: awmf_crosslingual_pubmedbert_k10


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[634]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


EVALUATION COMPLETE FOR awmf_crosslingual_pubmedbert_k10 -> Saved to Drive!
context_precision    0.001
context_recall       0.035
faithfulness         0.050
answer_relevancy     0.615
dtype: float64
